# Lee2019 SSVEP Fine-Tuning with SignalJEPA_PreLocal

Downstream fine-tuning on **Lee2019 SSVEP**.

- Dataset: Lee2019_SSVEP (MOABB), last 7 subjects
- EEG channels only; bandpass 0.5–40 Hz; resample to 128 Hz
- 5-fold stratified within-subject cross-validation
- Model: SignalJEPA_PreLocal
- Pretrained weights: Hugging Face 16s-60 checkpoint
- Fine-tuning scheme: **new-pre-local** (frozen pretrained backbone, train only new downstream layers)
- Training: Braindecode EEGClassifier (no manual loop)

Note: dataset split, preprocessing, downstream window, model family, and fine-tuning strategy follow the S-JEPA paper design. Optimizer hyperparameters in this notebook are implementation defaults.

# 1. Setup

## 1.1. Import Libraries

In [1]:
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime

import pandas as pd
import numpy as np
from numpy import multiply

import torch
import torch.nn as nn
from torch.utils.data import Subset

import mne

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    exponential_moving_standardize,
    create_windows_from_events,
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from skorch.callbacks import EarlyStopping
from skorch.helper import predefined_split

import builtins

from moabb.datasets import Lee2019_SSVEP

# Disable MNE logging
mne.set_log_level('WARNING')

print("All imports loaded successfully.")

All imports loaded successfully.


## 1.2. Runtime & Path Validation

In [2]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

NOTEBOOK_DIR = Path.cwd()
print(f"\nWorking directory: {NOTEBOOK_DIR}")

Runtime Environment:
  - Python: 3.11.14 | packaged by conda-forge | (main, Jan 27 2026, 00:01:01) [Clang 19.1.7 ]
  - Platform: macOS-26.2-arm64-arm-64bit

Working directory: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA


# 2. Configuration

## 2.1. Config

In [3]:
CONFIG = {
    # Paths
    "artifact_dir": str(NOTEBOOK_DIR / "artifacts" / "lee-2019-ssvep-fine-tuning"),

    # Reproducibility
    "seed": 42,

    # Preprocessing (paper-aligned)
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,
    "trial_duration_s": 4.19,

    # Classes (SSVEP)
    "n_classes": 4,

    # Cross-validation (paper-aligned structure)
    "cv_folds": 5,
    "val_split": 0.2,

    # Training (implementation defaults, not paper claims)
    "batch_size": 16,
    "learning_rate": 0.005,
    "n_epochs": 300,
    "early_stopping_patience": 50,

    # Pretrained weights (Hugging Face 16s-60 checkpoint)
    "pretrained_url": "https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth",

    # Device: "auto" | "cpu" | "cuda" | "mps"
    "device": "auto",
}

## 2.2. Create Artifact Directory

In [4]:
def create_run_id():
    # Generate unique run ID from timestamp + config hash.
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

In [5]:
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 2.3. Initialize Logger

In [6]:
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(arg) for arg in args)

    # Preserve visual spacing for prints like print("\n...")
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text) # type: ignore
            if flush:
                sys.__stdout__.flush() # type: ignore
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    # Apply leading blank lines first without timestamps
    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        # If only newlines were printed, preserve end behavior without adding a timestamp
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

## 2.4. Device Configuration

In [7]:
def resolve_device(device):
    """Resolve the compute device from config."""
    if device == "cpu":
        return torch.device("cpu")
    if device == "cuda":
        return torch.device("cuda")
    if device == "mps":
        return torch.device("mps")

    # "auto": prefer MPS > CUDA > CPU
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = resolve_device(CONFIG["device"])
print(f"Using device: {DEVICE}")

[2026-03-24 17:11:36] Using device: mps


## 2.5. Deterministic Seeding

In [8]:
def set_seed(seed):
    # Set all random seeds for reproducibility
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

## 2.6. Save Configuration

In [9]:
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")

[2026-03-24 17:11:36] ======================================================================
[2026-03-24 17:11:36] CONFIGURATION
[2026-03-24 17:11:36] ======================================================================
[2026-03-24 17:11:36]   artifact_dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning
[2026-03-24 17:11:36]   bandpass_high: 40.0
[2026-03-24 17:11:36]   bandpass_low: 0.5
[2026-03-24 17:11:36]   batch_size: 16
[2026-03-24 17:11:36]   cv_folds: 5
[2026-03-24 17:11:36]   device: auto
[2026-03-24 17:11:36]   early_stopping_patience: 50
[2026-03-24 17:11:36]   learning_rate: 0.005
[2026-03-24 17:11:36]   n_classes: 4
[2026-03-24 17:11:36]   n_epochs: 300
[2026-03-24 17:11:36]   pretrained_url: https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth
[2026-03-24 17:11:36]   seed: 42
[2026-03-24 17:11:36]   sfreq: 128
[2026-03-24 17:11:36]   trial_duration_s: 4.19
[2026-

# 3. Prepare Data

## 3.1. Subject Selection

In [10]:
def select_last_subjects(dataset, n_last=7):
    """Select the last n subjects required for downstream evaluation."""

    print("Dataset code:", dataset.code)
    print("Subjects Total:", len(dataset.subject_list))
    print("Events:", dataset.event_id)
    print("Interval:", dataset.interval)
    print("Paradigm:", dataset.paradigm)

    subjects = sorted(int(s) for s in dataset.subject_list)

    if len(subjects) < n_last:
        raise RuntimeError(f"Dataset has {len(subjects)} subjects, need at least {n_last}.")
    return subjects[-n_last:]

In [11]:
LEE_DATASET = Lee2019_SSVEP() # type: ignore
SUBJECTS = select_last_subjects(LEE_DATASET, n_last=7)

print(f"Selected subjects (last 7): {SUBJECTS}")
print(f"Expected evaluation folds: {len(SUBJECTS) * CONFIG['cv_folds']}")

print("\nBuilding MOABBDataset for Lee2019_SSVEP...")
DATASET = MOABBDataset(dataset_name="Lee2019_SSVEP", subject_ids=SUBJECTS)
print(f"MOABBDataset recordings loaded: {len(DATASET.datasets)}")

# SSVEP event mapping diagnostics
event_id = dict(sorted(LEE_DATASET.event_id.items(), key=lambda kv: kv[1]))
label_map = {code: idx for idx, (_, code) in enumerate(event_id.items())}
first_ann = sorted(set(DATASET.datasets[0].raw.annotations.description))
print(f"Discovered annotation descriptions (first recording): {first_ann}")
print(f"Dataset event_id mapping: {event_id}")
print(f"Label map (event_code -> class_index): {label_map}")

[2026-03-24 17:11:36] Dataset code: Lee2019-SSVEP
[2026-03-24 17:11:36] Subjects Total: 54
[2026-03-24 17:11:36] Events: {'12.0': 1, '8.57': 2, '6.67': 3, '5.45': 4}
[2026-03-24 17:11:36] Interval: [0.0, 4.0]
[2026-03-24 17:11:36] Paradigm: ssvep
[2026-03-24 17:11:36] Selected subjects (last 7): [48, 49, 50, 51, 52, 53, 54]
[2026-03-24 17:11:36] Expected evaluation folds: 35

[2026-03-24 17:11:36] Building MOABBDataset for Lee2019_SSVEP...
[2026-03-24 17:13:15] MOABBDataset recordings loaded: 14
[2026-03-24 17:13:15] Discovered annotation descriptions (first recording): [np.str_('12.0'), np.str_('5.45'), np.str_('6.67'), np.str_('8.57')]
[2026-03-24 17:13:15] Dataset event_id mapping: {'12.0': 1, '8.57': 2, '6.67': 3, '5.45': 4}
[2026-03-24 17:13:15] Label map (event_code -> class_index): {1: 0, 2: 1, 3: 2, 4: 3}


## 3.2. Derived Constants

In [12]:
def compute_epoch_window(sfreq, trial_duration_s):
    """Compute the exact downstream window and stop offset from CONFIG."""
    sfreq = float(sfreq)
    trial_duration_s = float(trial_duration_s)

    window_size_samples = round(trial_duration_s * sfreq)
    if window_size_samples <= 0:
        raise ValueError("Epoch window must contain at least one sample.")

    base_trial_duration_s = 4.0
    base_trial_samples = round(base_trial_duration_s * sfreq)
    trial_stop_offset_samples = window_size_samples - base_trial_samples
    effective_duration = window_size_samples / sfreq

    if trial_stop_offset_samples < 0:
        raise ValueError(
            f"Requested window ({window_size_samples}) is shorter than base trial ({base_trial_samples})."
        )

    print("Epoch window configuration:")
    print(f"  Target sfreq:                {sfreq:.0f} Hz")
    print(f"  Requested duration:          {trial_duration_s:.6f} s")
    print(f"  Window samples (rounded):    {window_size_samples}")
    print(f"  Base trial samples (4.0 s):  {base_trial_samples}")
    print(f"  Trial stop offset samples:   {trial_stop_offset_samples}")
    print(f"  Effective window duration:   {effective_duration:.6f} s")

    return window_size_samples, base_trial_samples, trial_stop_offset_samples

In [13]:
WINDOW_SAMPLES, BASE_TRIAL_SAMPLES, TRIAL_STOP_OFFSET_SAMPLES = compute_epoch_window(
    sfreq=CONFIG["sfreq"],
    trial_duration_s=CONFIG["trial_duration_s"],
)

[2026-03-24 17:13:16] Epoch window configuration:
[2026-03-24 17:13:16]   Target sfreq:                128 Hz
[2026-03-24 17:13:16]   Requested duration:          4.190000 s
[2026-03-24 17:13:16]   Window samples (rounded):    536
[2026-03-24 17:13:16]   Base trial samples (4.0 s):  512
[2026-03-24 17:13:16]   Trial stop offset samples:   24
[2026-03-24 17:13:16]   Effective window duration:   4.187500 s


## 3.3. Filter and Resample Data

In [14]:
def get_braindecode_preprocessors(sfreq, bandpass_low, bandpass_high):
    """Build Braindecode preprocessors in memory-efficient order.

    Preprocessing steps:
      1. Pick EEG channels.
      2. Convert V -> uV (scale raw voltage to microvolts).
      3. Resample BEFORE filtering — MNE's resample applies anti-aliasing internally,
         so doing this first reduces the number of samples that the filter must
         process, cutting both memory usage and compute time significantly.
      4. Bandpass filter on the already-downsampled signal.
      5. Exponential moving standardisation for per-recording normalisation.
    """
    factor = 1e6  # V -> uV

    preprocessors = [
        # Step 1: select EEG channels
        Preprocessor("pick", picks="eeg"),
        # Step 2: convert raw voltage to microvolts
        Preprocessor(lambda data: multiply(data, factor)),
        # Step 3: resample BEFORE filtering — reduces data size before the filter
        # pass, keeping peak memory and compute cost low on local machines.
        Preprocessor("resample", sfreq=sfreq),
        # Step 4: bandpass filter on the downsampled signal
        Preprocessor("filter", l_freq=bandpass_low, h_freq=bandpass_high),
        # Step 5: per-recording online normalisation
        Preprocessor(
            exponential_moving_standardize,
            factor_new=1e-3,
            init_block_size=1000,
        ),
    ]

    return preprocessors

## 3.4. Channel Info and Validation

In [15]:
def get_channel_info_from_one_recording(dataset):
    """Extract channel metadata from the first preprocessed recording."""
    first_raw = dataset.datasets[0].raw
    return first_raw.info["chs"], len(first_raw.ch_names), list(first_raw.ch_names)

In [16]:
def validate_channel_consistency(dataset):
    """Validate all recordings share channel count/order after preprocessing."""
    checked_runs = 0
    unique_channel_counts = set()
    reference_names = None
    inconsistent_runs = []

    for ds in dataset.datasets:
        checked_runs += 1
        raw = ds.raw
        ch_names = tuple(raw.ch_names)
        unique_channel_counts.add(len(ch_names))

        if reference_names is None:
            reference_names = ch_names
            continue

        if ch_names != reference_names:
            inconsistent_runs.append(
                {
                    "run": str(ds.description.to_dict()),
                    "n_channels": len(ch_names),
                    "first_channels": list(ch_names[:10]),
                }
            )

    channel_order_consistent = len(inconsistent_runs) == 0
    print("Channel consistency summary:")
    print(f"  Checked runs: {checked_runs}")
    print(f"  Unique channel counts: {sorted(unique_channel_counts)}")
    print(f"  Channel order consistent: {channel_order_consistent}")

    if inconsistent_runs:
        print(f"  Inconsistent examples: {inconsistent_runs[:2]}")

    return {
        "checked_runs": checked_runs,
        "unique_channel_counts": sorted(unique_channel_counts),
        "channel_order_consistent": channel_order_consistent,
    }

In [17]:
print("Applying Braindecode preprocessors to MOABBDataset...")
preprocessors = get_braindecode_preprocessors(
    sfreq=CONFIG["sfreq"],
    bandpass_low=CONFIG["bandpass_low"],
    bandpass_high=CONFIG["bandpass_high"],
)

# n_jobs=1 is intentional: each parallel worker would load a full copy of the
# dataset into memory, causing large RAM spikes on local machines.
# Sequential processing (n_jobs=1) is slower but keeps peak memory manageable.
preprocess(DATASET, preprocessors, n_jobs=1)
print("Preprocessing complete.")

# --- Post-preprocessing validation ---
print("\nPost-preprocessing validation")
print(f"  Recordings in DATASET: {len(DATASET.datasets)}")
subject_ids = sorted(set(str(ds.description["subject"]) for ds in DATASET.datasets))
print(f"  Unique subject IDs:    {subject_ids}")
first_raw = DATASET.datasets[0].raw
print(f"  sfreq (first rec):     {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:     {len(first_raw.ch_names)}")
print(f"  First 10 channels:     {list(first_raw.ch_names[:10])}")

[2026-03-24 17:13:16] Applying Braindecode preprocessors to MOABBDataset...


/Users/vadim/miniforge3/envs/dl/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:76: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(
/Users/vadim/miniforge3/envs/dl/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:74: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


[2026-03-24 17:14:18] Preprocessing complete.

[2026-03-24 17:14:18] Post-preprocessing validation
[2026-03-24 17:14:18]   Recordings in DATASET: 14
[2026-03-24 17:14:18]   Unique subject IDs:    ['48', '49', '50', '51', '52', '53', '54']
[2026-03-24 17:14:18]   sfreq (first rec):     128.0 Hz
[2026-03-24 17:14:18]   EEG channel count:     62
[2026-03-24 17:14:18]   First 10 channels:     ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2']


In [18]:
CHANNEL_VALIDATION_SUMMARY = validate_channel_consistency(DATASET)
CHS_INFO, N_CHANNELS, CH_NAMES = get_channel_info_from_one_recording(DATASET)

print(f"Validated EEG channels: {N_CHANNELS}")
print(f"First 10 EEG channels: {CH_NAMES[:10]}")

[2026-03-24 17:14:18] Channel consistency summary:
[2026-03-24 17:14:18]   Checked runs: 14
[2026-03-24 17:14:18]   Unique channel counts: [62]
[2026-03-24 17:14:18]   Channel order consistent: True
[2026-03-24 17:14:18] Validated EEG channels: 62
[2026-03-24 17:14:18] First 10 EEG channels: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2']


## 3.5. Subject Preprocessing

In [19]:
def build_windows_dataset(dataset, window_size_samples, trial_stop_offset_samples):
    """Create event-based windows from preprocessed MOABBDataset."""
    trial_start_offset_samples = 0

    print(f"Requested window samples: {window_size_samples}")
    print(f"Base trial samples:       {BASE_TRIAL_SAMPLES}")
    print(f"Stop offset samples:      {trial_stop_offset_samples}")

    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=trial_stop_offset_samples,
        window_size_samples=window_size_samples,
        window_stride_samples=window_size_samples,
        drop_last_window=False,
        preload=True,
    )

    return windows_dataset

In [20]:
def get_targets_from_dataset(dataset):
    """Extract integer labels from a Braindecode dataset/subset."""
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

In [21]:
def get_subject_windows(windows_dataset, subject_ids):
    """Split windows by subject and ensure all requested subjects are available."""
    split_by_subject = windows_dataset.split("subject")
    subject_windows = {}

    for sid in subject_ids:
        key = str(sid)
        if key in split_by_subject:
            subject_windows[sid] = split_by_subject[key]
        elif sid in split_by_subject:
            subject_windows[sid] = split_by_subject[sid]
        else:
            raise RuntimeError(f"Subject {sid} not found in windows split keys: {list(split_by_subject.keys())[:10]}")

    return subject_windows

In [22]:
WINDOWS_DATASET = build_windows_dataset(
    DATASET,
    window_size_samples=WINDOW_SAMPLES,
    trial_stop_offset_samples=TRIAL_STOP_OFFSET_SAMPLES,
)

# Global label diagnostics
ALL_LABELS = get_targets_from_dataset(WINDOWS_DATASET)
all_counts = np.bincount(ALL_LABELS, minlength=CONFIG["n_classes"])
print(f"Global class counts: {all_counts.tolist()}")
print(f"Chance level ({CONFIG['n_classes']}-class): {1.0 / CONFIG['n_classes']:.2f}")

# Window/sample diagnostics
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
y0 = sample0[1]
print(f"One window shape: {tuple(x0.shape)}")
print(f"One window label: {int(y0)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")

SUBJECT_WINDOWS = get_subject_windows(WINDOWS_DATASET, SUBJECTS)
print(f"Subjects in window split: {list(SUBJECT_WINDOWS.keys())}")

[2026-03-24 17:14:18] Requested window samples: 536
[2026-03-24 17:14:18] Base trial samples:       512
[2026-03-24 17:14:18] Stop offset samples:      24
[2026-03-24 17:14:18] Global class counts: [350, 350, 350, 350]
[2026-03-24 17:14:18] Chance level (4-class): 0.25
[2026-03-24 17:14:18] One window shape: (62, 536)
[2026-03-24 17:14:18] One window label: 2
[2026-03-24 17:14:18] Window sample length expected=536, got=536
[2026-03-24 17:14:18] Subjects in window split: [48, 49, 50, 51, 52, 53, 54]


In [23]:
def summarize_subject_windows(subject_windows, n_classes):
    """Summarize per-subject window counts and class balance."""
    rows = []
    for subject_id in sorted(subject_windows):
        ds = subject_windows[subject_id]
        y = get_targets_from_dataset(ds)
        counts = np.bincount(y, minlength=n_classes)
        sample = ds[0]
        x = sample[0]

        print(
            f"  Subject {subject_id}: n_windows={len(ds)}, "
            f"window_shape={tuple(x.shape)}, class_counts={counts.tolist()}"
        )
        rows.append(
            {
                "subject_id": int(subject_id),
                "n_windows": int(len(ds)),
                "n_channels": int(x.shape[0]),
                "n_times": int(x.shape[1]),
                "class_counts": counts.tolist(),
            }
        )

    return pd.DataFrame(rows)

In [24]:
print("Summarizing per-subject windows...")
SUBJECT_TRIALS_SUMMARY = summarize_subject_windows(SUBJECT_WINDOWS, CONFIG["n_classes"])
print(SUBJECT_TRIALS_SUMMARY)

# Keep compatibility with downstream reporting code paths.
SUBJECT_TRIALS = {sid: (None, get_targets_from_dataset(ds)) for sid, ds in SUBJECT_WINDOWS.items()}

[2026-03-24 17:14:18] Summarizing per-subject windows...
[2026-03-24 17:14:18]   Subject 48: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]   Subject 49: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]   Subject 50: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]   Subject 51: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]   Subject 52: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]   Subject 53: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]   Subject 54: n_windows=200, window_shape=(62, 536), class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:18]    subject_id  n_windows  n_channels  n_times      class_counts
0          48        200          62      536  [50, 50, 50, 50]
1          49        200          62      536  [50, 50, 50

## 3.6. Validation and Summary

In [25]:
def inspect_preprocessing_sanity(windows_dataset):
    """Quick sanity checks on one window after preprocessing."""
    sample = windows_dataset[0]
    x = sample[0]
    y = sample[1]
    x_np = np.asarray(x)

    print("Preprocessing sanity checks:")
    print(f"  One sample label: {int(y)}")
    print(f"  One sample shape: {tuple(x_np.shape)}")
    print(f"  Mean(abs(signal)): {float(np.mean(np.abs(x_np))):.4f}")
    print(f"  Mean(signal):      {float(np.mean(x_np)):.4f}")
    print(f"  Std(signal):       {float(np.std(x_np)):.4f}")

In [26]:
print("Window validation summary:")

assert len(WINDOWS_DATASET) > 0, "No windows created."
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
assert x0.shape[-1] == WINDOW_SAMPLES, (
    f"Window length mismatch: got {x0.shape[-1]} expected {WINDOW_SAMPLES}"
)
assert WINDOW_SAMPLES == 536, f"Expected 536 MI samples, got {WINDOW_SAMPLES}"

all_labels = get_targets_from_dataset(WINDOWS_DATASET)
assert np.isin(all_labels, np.arange(CONFIG["n_classes"])).all(), (
    f"Invalid labels found: {np.unique(all_labels)}"
)

for sid, ds in SUBJECT_WINDOWS.items():
    y_sid = get_targets_from_dataset(ds)
    counts = np.bincount(y_sid, minlength=CONFIG["n_classes"])
    print(f"  Subject {sid}: n_windows={len(ds)} class_counts={counts.tolist()}")

effective_duration = WINDOW_SAMPLES / CONFIG["sfreq"]

print(f"\nValidated sampling rate target: {CONFIG['sfreq']} Hz")
print(f"Validated window sample count:   {WINDOW_SAMPLES}")
print(f"Validated window duration:       {effective_duration:.4f} s")
print(f"Validated channel count:         {N_CHANNELS}")
print(f"Total windows retained:          {len(WINDOWS_DATASET)}")

inspect_preprocessing_sanity(WINDOWS_DATASET)

[2026-03-24 17:14:18] Window validation summary:
[2026-03-24 17:14:19]   Subject 48: n_windows=200 class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Subject 49: n_windows=200 class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Subject 50: n_windows=200 class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Subject 51: n_windows=200 class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Subject 52: n_windows=200 class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Subject 53: n_windows=200 class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Subject 54: n_windows=200 class_counts=[50, 50, 50, 50]

[2026-03-24 17:14:19] Validated sampling rate target: 128 Hz
[2026-03-24 17:14:19] Validated window sample count:   536
[2026-03-24 17:14:19] Validated window duration:       4.1875 s
[2026-03-24 17:14:19] Validated channel count:         62
[2026-03-24 17:14:19] Total windows retained:          1400
[2026-03-24 17:14:19] Preprocessing sanity checks:
[2026-03-24 17:14:19]   One sample la

# 4. Model

## 4.1. Build Model

In [27]:
def build_model():
    """Instantiate SignalJEPA_PreLocal."""
    # actual window duration
    trial_s = WINDOW_SAMPLES / CONFIG["sfreq"]
    model = SignalJEPA_PreLocal(
        sfreq=CONFIG["sfreq"],
        input_window_seconds=trial_s,
        chs_info=CHS_INFO,
        n_outputs=CONFIG["n_classes"],
    )
    return model

In [28]:
# Verify once that the model builds without error
test_model = build_model()
total_p = sum(p.numel() for p in test_model.parameters())
trainable_p = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print("SignalJEPA_PreLocal instantiated successfully.")
print(f"  Total parameters:     {total_p:,}")
print(f"  Trainable parameters: {trainable_p:,}")
print(test_model)
del test_model

[2026-03-24 17:14:19] SignalJEPA_PreLocal instantiated successfully.
[2026-03-24 17:14:19]   Total parameters:     18,192
[2026-03-24 17:14:19]   Trainable parameters: 18,192
[2026-03-24 17:14:19] ======================================================================================================================================================
Layer (type (var_name):depth-idx)                  Input Shape               Output Shape              Param #                   Kernel Shape
SignalJEPA_PreLocal (SignalJEPA_PreLocal)          [1, 62, 536]              [1, 4]                    --                        --
├─Sequential (spatial_conv): 1-1                   [1, 62, 536]              [1, 4, 536]               --                        --
│    └─Rearrange (0): 2-1                          [1, 62, 536]              [1, 1, 62, 536]           --                        --
│    └─Conv2d (1): 2-2                             [1, 1, 62, 536]           [1, 4, 1, 536]            252        

## 4.2. Load Pretrained Weights from Hugging Face

In [29]:
def download_pretrained_state_dict(url):
    """Download Signal-JEPA pretrained weights from Hugging Face."""
    print("Initializing from 16s-60 pretrained checkpoint (Hugging Face):")
    print(f"  {url}")
    sd = torch.hub.load_state_dict_from_url(url, map_location="cpu")
    print(f"  Downloaded {len(sd)} keys")
    return sd

In [30]:
def load_pretrained_weights(model, state_dict):
    """
    Filter transformer and pos_encoder keys (not used by PreLocal downstream) and load encoder weights.
    
    Strictly validates that only expected downstream layers are missing from the pretrained weights.
    Raises an error if unexpected keys remain or if missing keys do not match downstream layers.
    """
    filtered = {
        k: v for k, v in state_dict.items()
        if not (k.startswith("transformer.") or k.startswith("pos_encoder."))
    }
    n_filtered_transformer = sum(1 for k in state_dict if k.startswith("transformer."))
    n_filtered_pos_encoder = sum(1 for k in state_dict if k.startswith("pos_encoder."))
    
    missing_keys, unexpected_keys = model.load_state_dict(filtered, strict=False)
    
    print("Pretrained weight loading:")
    print(f"  Downloaded keys:        {len(state_dict)}")
    print(f"  Filtered (transformer): {n_filtered_transformer}")
    print(f"  Filtered (pos_encoder): {n_filtered_pos_encoder}")
    print(f"  Keys to load:           {len(filtered)}")
    print(f"  Missing keys:           {len(missing_keys)}")
    print(f"  Unexpected keys:        {len(unexpected_keys)}")
    print(f"  Missing preview:        {missing_keys[:6]}")
    print(f"  Unexpected preview:     {unexpected_keys[:6]}")

    return {
        "downloaded_keys": len(state_dict),
        "filtered_transformer": n_filtered_transformer,
        "filtered_pos_encoder": n_filtered_pos_encoder,
        "loaded_keys": len(filtered),
        "missing_keys": list(missing_keys),
        "unexpected_keys": list(unexpected_keys),
    }

In [31]:
# Download once — reused for every fold
PRETRAINED_STATE_DICT = download_pretrained_state_dict(CONFIG["pretrained_url"])

[2026-03-24 17:14:19] Initializing from 16s-60 pretrained checkpoint (Hugging Face):
[2026-03-24 17:14:19]   https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth
[2026-03-24 17:14:19]   Downloaded 179 keys


## 4.3. Freeze Encoder

In [32]:
def freeze_encoder(model):
    """
    Freeze all encoder layers; keep only spatial_conv and final_layer trainable.
    
    Implements 'new-pre-local' strategy: all pretrained components frozen,
    only the newly introduced downstream layers trained.
    """
    for name, param in model.named_parameters():
        param.requires_grad = False
    
    trainable_names = []
    for name, param in model.named_parameters():
        if 'spatial_conv' in name or 'final_layer' in name:
            param.requires_grad = True
            trainable_names.append(name)
    
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("Encoder freeze configuration:")
    print(f"  Total parameters:        {total:,}")
    print(f"  Trainable parameters:    {trainable:,}")
    print(f"  Training percentage:     {100 * trainable / total:.1f}%")
    print(f"  Trainable layers: {len(trainable_names)}")
    for tname in sorted(trainable_names):
        print(f"    - {tname}")
    
    assert all(("spatial_conv" in n or "final_layer" in n) for n in trainable_names), (
        "Found trainable parameters outside spatial_conv/final_layer."
    )

# 5. Training

## 5.1. EEGClassifier Fold Runner

In [33]:
def run_single_fold(fold_id, subject_id, subject_dataset, idx_train, idx_val, idx_test):
    """Train and evaluate one fold using EEGClassifier on dataset subsets."""
    train_set = Subset(subject_dataset, idx_train.tolist())
    valid_set = Subset(subject_dataset, idx_val.tolist())
    test_set = Subset(subject_dataset, idx_test.tolist())

    y_all = get_targets_from_dataset(subject_dataset)
    y_train = y_all[idx_train]
    y_val = y_all[idx_val]
    y_test = y_all[idx_test]

    train_counts = np.bincount(y_train, minlength=CONFIG["n_classes"])
    val_counts = np.bincount(y_val, minlength=CONFIG["n_classes"])
    test_counts = np.bincount(y_test, minlength=CONFIG["n_classes"])

    print(f"  Subject {subject_id} Fold {fold_id}: n_train={len(train_set)} n_val={len(valid_set)} n_test={len(test_set)}")
    print(f"    Train class counts: {train_counts.tolist()}")
    print(f"    Val class counts:   {val_counts.tolist()}")
    print(f"    Test class counts:  {test_counts.tolist()}")

    model = build_model()
    load_pretrained_weights(model, PRETRAINED_STATE_DICT)
    freeze_encoder(model)

    callbacks = [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=CONFIG["early_stopping_patience"],
                lower_is_better=True,
                load_best=True,
            ),
        ),
    ]

    clf = EEGClassifier(
        model,
        criterion=nn.CrossEntropyLoss,
        optimizer=torch.optim.AdamW,
        optimizer__lr=CONFIG["learning_rate"],
        batch_size=CONFIG["batch_size"],
        max_epochs=CONFIG["n_epochs"],
        device=DEVICE,
        callbacks=callbacks,
        train_split=predefined_split(valid_set),
        iterator_train__shuffle=True,
        classes=range(CONFIG["n_classes"]),
    )

    clf.fit(train_set, y=None)
    y_pred = clf.predict(test_set)

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0 # type: ignore
    valid_loss_curve = [
        (int(row["epoch"]), float(row["valid_loss"]))
        for row in clf.history
        if "valid_loss" in row
    ]

    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])  # lower is better
    else:
        best_epoch, best_valid_loss = None, None

    acc = float(accuracy_score(y_test, y_pred))
    bal_acc = float(balanced_accuracy_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(CONFIG["n_classes"])).tolist()
    pred_hist = np.bincount(y_pred, minlength=CONFIG["n_classes"]).tolist()

    print(f"    Stopped epoch:            {stopped_epoch}")
    print(f"    Best epoch (valid_loss):  {best_epoch}")
    print(f"    Best valid_loss:          {best_valid_loss}")
    print(f"    Test accuracy (primary):  {acc:.4f}")
    print(f"    Test balanced acc:        {bal_acc:.4f}")
    print(f"    Confusion matrix:         {cm}")
    print(f"    Prediction histogram:     {pred_hist}")

    return {
        "subject_id": str(subject_id),
        "fold_id": fold_id,
        "n_train": len(train_set),
        "n_val": len(valid_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "val_class_counts": val_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }

## 5.2. Subject Cross-Validation Runner

In [34]:
def make_fold_splits(y, n_folds, val_split, seed, n_classes):
    """Create stratified k-fold index splits for one subject."""
    indices = np.arange(len(y))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []

    for fold_id, (tv_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        y_tv = y[tv_idx]
        y_test_f = y[test_idx]

        tr_local_idx, val_local_idx = train_test_split(
            np.arange(len(tv_idx)),
            test_size=val_split,
            stratify=y_tv,
            random_state=seed + fold_id,
        )

        train_idx = tv_idx[tr_local_idx]
        val_idx = tv_idx[val_local_idx]

        train_counts = np.bincount(y[train_idx], minlength=n_classes)
        val_counts = np.bincount(y[val_idx], minlength=n_classes)
        test_counts = np.bincount(y_test_f, minlength=n_classes)

        print(f"  Fold {fold_id} split class counts:")
        print(f"    train={train_counts.tolist()} val={val_counts.tolist()} test={test_counts.tolist()}")

        folds.append(
            {
                "fold_id": fold_id,
                "idx_train": train_idx,
                "idx_val": val_idx,
                "idx_test": test_idx,
            }
        )

    return folds

In [35]:
def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds, val_split, seed):
    """Run within-subject stratified CV for one subject."""
    y = get_targets_from_dataset(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows class_counts={counts.tolist()}")

    folds = make_fold_splits(
        y,
        n_folds=cv_folds,
        val_split=val_split,
        seed=seed,
        n_classes=n_classes,
    )
    results = []

    for fold in folds:
        result = run_single_fold(
            fold["fold_id"],
            subject_id,
            subject_dataset,
            fold["idx_train"],
            fold["idx_val"],
            fold["idx_test"],
        )
        results.append(result)
        print(
            f"    fold_acc={result['accuracy']:.4f} "
            f"fold_bal_acc={result['balanced_accuracy']:.4f} "
            f"cm={result['confusion_matrix']}"
        )

    mean_acc = float(np.mean([r["accuracy"] for r in results]))
    print(f"  Subject {subject_id} mean accuracy: {mean_acc:.4f}")

    return results

## 5.3. Run All Subjects

In [36]:
print("=" * 70)
print("STARTING CROSS-VALIDATION")
print("=" * 70)
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print(f"Batch size:    {CONFIG['batch_size']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print("=" * 70)

set_seed(CONFIG["seed"])
FOLD_RESULTS = []

for sid in SUBJECT_WINDOWS:
    subject_ds = SUBJECT_WINDOWS[sid]
    fold_results = run_subject_cv(
        sid,
        subject_ds,
        CONFIG["n_classes"],
        CONFIG["cv_folds"],
        CONFIG["val_split"],
        CONFIG["seed"],
    )
    FOLD_RESULTS.extend(fold_results)

print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

[2026-03-24 17:14:19] ======================================================================
[2026-03-24 17:14:19] STARTING CROSS-VALIDATION
[2026-03-24 17:14:19] ======================================================================
[2026-03-24 17:14:19] Subjects:      [48, 49, 50, 51, 52, 53, 54]
[2026-03-24 17:14:19] CV folds:      5
[2026-03-24 17:14:19] Learning rate: 0.005
[2026-03-24 17:14:19] Batch size:    16
[2026-03-24 17:14:19] Max epochs:    300
[2026-03-24 17:14:19] Device:        mps
[2026-03-24 17:14:19] ======================================================================

[2026-03-24 17:14:19] Subject 48: 200 windows class_counts=[50, 50, 50, 50]
[2026-03-24 17:14:19]   Fold 1 split class counts:
[2026-03-24 17:14:19]     train=[32, 32, 32, 32] val=[8, 8, 8, 8] test=[10, 10, 10, 10]
[2026-03-24 17:14:19]   Fold 2 split class counts:
[2026-03-24 17:14:19]     train=[32, 32, 32, 32] val=[8, 8, 8, 8] test=[10, 10, 10, 10]
[2026-03-24 17:14:19]   Fold 3 split class count

# 6. Results

## 6.1. Aggregate Metrics

In [37]:
# Per-subject aggregation
SUBJECT_METRICS = {}

for result in FOLD_RESULTS:
    sid = result["subject_id"]
    if sid not in SUBJECT_METRICS:
        SUBJECT_METRICS[sid] = {"accuracies": [], "balanced_accuracies": []}
    SUBJECT_METRICS[sid]["accuracies"].append(result["accuracy"])
    SUBJECT_METRICS[sid]["balanced_accuracies"].append(result["balanced_accuracy"])

for sid, m in SUBJECT_METRICS.items():
    m["mean_accuracy"] = float(np.mean(m["accuracies"]))
    m["std_accuracy"] = float(np.std(m["accuracies"]))
    m["mean_balanced_accuracy"] = float(np.mean(m["balanced_accuracies"]))
    m["std_balanced_accuracy"] = float(np.std(m["balanced_accuracies"]))

# Global aggregation
all_accs = [r["accuracy"] for r in FOLD_RESULTS]
all_bal_accs = [r["balanced_accuracy"] for r in FOLD_RESULTS]

GLOBAL_METRICS = {
    "mean_accuracy": float(np.mean(all_accs)),
    "std_accuracy": float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_bal_accs)),
    "std_balanced_accuracy": float(np.std(all_bal_accs)),
    "n_subjects": len(SUBJECT_METRICS),
    "n_folds_total": len(FOLD_RESULTS),
}

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for sid, m in sorted(SUBJECT_METRICS.items()):
    print(f"  Subject {sid}:  acc={m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}  "
          f"bal_acc={m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}")
print("-" * 70)
print(f"  OVERALL:      acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  "
      f"bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

[2026-03-24 17:29:55] ======================================================================
[2026-03-24 17:29:55] AGGREGATED RESULTS
[2026-03-24 17:29:55] ======================================================================
[2026-03-24 17:29:55]   Subject 48:  acc=0.9850±0.0200  bal_acc=0.9850±0.0200
[2026-03-24 17:29:55]   Subject 49:  acc=0.9800±0.0187  bal_acc=0.9800±0.0187
[2026-03-24 17:29:55]   Subject 50:  acc=0.9850±0.0200  bal_acc=0.9850±0.0200
[2026-03-24 17:29:55]   Subject 51:  acc=0.4400±0.1914  bal_acc=0.4400±0.1914
[2026-03-24 17:29:55]   Subject 52:  acc=0.9700±0.0292  bal_acc=0.9700±0.0292
[2026-03-24 17:29:55]   Subject 53:  acc=0.9750±0.0224  bal_acc=0.9750±0.0224
[2026-03-24 17:29:55]   Subject 54:  acc=0.6650±0.0539  bal_acc=0.6650±0.0539
[2026-03-24 17:29:55] ----------------------------------------------------------------------
[2026-03-24 17:29:55]   OVERALL:      acc=0.8571±0.2163  bal_acc=0.8571±0.2163
[2026-03-24 17:29:55] =================================

## 6.2. Save Outputs

In [38]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, 'w') as f:
    json.dump(FOLD_RESULTS, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, 'w') as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
print(f"Subject metrics saved to: {subject_metrics_path}")

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, 'w') as f:
    json.dump(GLOBAL_METRICS, f, indent=2)
print(f"Global metrics saved to:  {global_metrics_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "subjects": [str(s) for s in SUBJECT_TRIALS.keys()],
    "model": "SignalJEPA_PreLocal",
    "pretrained_url": CONFIG["pretrained_url"],
    "global_metrics": GLOBAL_METRICS,
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, 'w') as f:
    json.dump(run_metadata, f, indent=2)
print(f"Run metadata saved to:    {run_metadata_path}")

print(f"\nAll artifacts in: {ARTIFACT_DIR}")

[2026-03-24 17:29:55] CV results saved to:      /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning/20260324_1711_256d68ab/cv_results.json
[2026-03-24 17:29:55] Subject metrics saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning/20260324_1711_256d68ab/subject_metrics.json
[2026-03-24 17:29:55] Global metrics saved to:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning/20260324_1711_256d68ab/global_metrics.json
[2026-03-24 17:29:55] Run metadata saved to:    /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning/20260324_1711_256d68ab/run_metadata.json

[2026-03-24 17:29:55] All artifacts in: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning/20260324_1

## 6.3. Final Summary

In [39]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:    {RUN_ID}")
print(f"Artifacts: {ARTIFACT_DIR}")
print()
print("Configuration:")
print(f"  Subjects (last 7): {list(SUBJECT_TRIALS.keys())}")
print(f"  Bandpass:          {CONFIG['bandpass_low']}–{CONFIG['bandpass_high']} Hz")
print(f"  Resample:          {CONFIG['sfreq']} Hz")
print(f"  CV folds:          {CONFIG['cv_folds']}")
print(f"  Learning rate:     {CONFIG['learning_rate']}")
print(f"  Batch size:        {CONFIG['batch_size']}")
print(f"  Max epochs:        {CONFIG['n_epochs']}")
print()
print("Results:")
print(f"  Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"  Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")


[2026-03-24 17:29:55] ======================================================================
[2026-03-24 17:29:55] EXPERIMENT SUMMARY
[2026-03-24 17:29:55] ======================================================================
[2026-03-24 17:29:55] Run ID:    20260324_1711_256d68ab
[2026-03-24 17:29:55] Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/lee-2019-ssvep-fine-tuning/20260324_1711_256d68ab

[2026-03-24 17:29:55] Configuration:
[2026-03-24 17:29:55]   Subjects (last 7): [48, 49, 50, 51, 52, 53, 54]
[2026-03-24 17:29:55]   Bandpass:          0.5–40.0 Hz
[2026-03-24 17:29:55]   Resample:          128 Hz
[2026-03-24 17:29:55]   CV folds:          5
[2026-03-24 17:29:55]   Learning rate:     0.005
[2026-03-24 17:29:55]   Batch size:        16
[2026-03-24 17:29:55]   Max epochs:        300

[2026-03-24 17:29:55] Results:
[2026-03-24 17:29:55]   Mean Accuracy:          0.8571 ± 0.2163
[2026-03-24 17:29:55]   Mean Balanced Accura